# 📊 01 — Analyse Exploratoire (EDA) — Social Segmentation Clustering

**Objectif** : Comprendre la structure des données clients avant tout modèle de clustering (K-Means / DBSCAN / Agglomératif) :
distribution des variables, saisonnalité, profils démographiques, valeurs manquantes ou aberrantes.

**Données attendues** : fichier CSV dans `data/raw/Segmentation Data .csv`

**Colonnes** :
| Colonne | Type | Description |
|---------|------|-------------|
| `ID` | UUID | Identifiant unique client |
| `Age` | int | Âge du client |
| `Gender` | str | Male / Female |
| `Income` | int | Revenu annuel (€ ou $) |
| `Score` | int | Score comportemental (0–100) |

**Convention notebooks** : `{ordre}_{type}_{sujet}.ipynb`  
Ce notebook : `01_eda_social_segmentation.ipynb`

---

## 0️⃣ Chemins & Configuration

In [ ]:
from pathlib import Path

# Racine du projet (dossier au-dessus de notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
RESULTS        = ROOT / 'results'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

print('Racine projet      :', ROOT.resolve())
print('Fichiers CSV bruts :', sorted(p.name for p in DATA_RAW.glob('*.csv')))

## 1️⃣ Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Détection automatique du fichier CSV
candidates = ['Segmentation Data .csv', 'segmentation_data.csv', 'data.csv']
csv_paths  = [DATA_RAW / name for name in candidates if (DATA_RAW / name).exists()]

if not csv_paths:
    others = sorted(DATA_RAW.glob('*.csv'))
    if not others:
        raise FileNotFoundError(f'Aucun CSV dans {DATA_RAW.resolve()} — copier le fichier dans data/raw/')
    csv_paths = [others[0]]

CSV_PATH = csv_paths[0]
print('Fichier chargé :', CSV_PATH.name)

# Chargement (pas d'en-tête dans le CSV original)
df = pd.read_csv(
    CSV_PATH,
    header=None,
    names=['ID', 'Age', 'Gender', 'Income', 'Score']
)

display(df.head(10))
print('\nDimensions :', df.shape)
print('Types      :\n', df.dtypes)
print('\nNaN par colonne :\n', df.isna().sum())

## 2️⃣ Statistiques descriptives

In [ ]:
print('=== Statistiques numériques ===')
display(df[['Age', 'Income', 'Score']].describe().round(2))

print('\n=== Répartition Gender ===')
gender_counts = df['Gender'].value_counts()
gender_pct    = df['Gender'].value_counts(normalize=True).mul(100).round(1)
display(pd.DataFrame({'Nombre': gender_counts, 'Pourcentage (%)': gender_pct}))

## 3️⃣ Contrôle qualité — Doublons, Valeurs manquantes, Outliers

In [ ]:
from scipy import stats

# ── 3a. Doublons sur ID ──────────────────────────────────────────────────────
dup_id   = df['ID'].duplicated(keep=False)
dup_rows = df.duplicated(keep=False)

print('=== Doublons ===')
print(f'IDs dupliqués          : {dup_id.sum()}')
print(f'Lignes entièrement en double : {dup_rows.sum()}')
if dup_id.sum():
    display(df.loc[dup_id].head(10))

# ── 3b. Valeurs manquantes ───────────────────────────────────────────────────
print('\n=== Valeurs manquantes ===')
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else 'Aucune valeur manquante ✔')

# ── 3c. Plages de valeurs (cohérence métier) ─────────────────────────────────
print('\n=== Plages de valeurs ===')
for col in ['Age', 'Income', 'Score']:
    print(f'  {col:<8} min={df[col].min():>8}  max={df[col].max():>8}')

# ── 3d. Outliers — méthode IQR × 3 ──────────────────────────────────────────
print('\n=== Outliers (IQR × 3) ===')
for col in ['Age', 'Income', 'Score']:
    q1, q3  = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr     = q3 - q1
    lo, hi  = q1 - 3 * iqr, q3 + 3 * iqr
    out     = df[(df[col] < lo) | (df[col] > hi)]
    print(f'  {col:<8} : {len(out)} outliers  (bornes [{lo:.1f}, {hi:.1f}])')

# ── 3e. Tests de normalité (Shapiro-Wilk) ────────────────────────────────────
print('\n=== Tests de normalité Shapiro-Wilk ===')
for col in ['Age', 'Income', 'Score']:
    sample = df[col].sample(min(500, len(df)), random_state=42)
    _, p   = stats.shapiro(sample)
    label  = '✔ normale' if p > 0.05 else '✘ non-normale'
    print(f'  {col:<8} p={p:.4f}  → {label}')

## 4️⃣ Visualisations — Distributions individuelles

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cols   = ['Age', 'Income', 'Score']
colors = ['#3498db', '#2ecc71', '#e74c3c']

for ax, col, color in zip(axes, cols, colors):
    sns.histplot(df[col], bins=30, kde=True, color=color, ax=ax, edgecolor='white')
    ax.axvline(df[col].mean(),   color='black',  linestyle='--', lw=1.5, label=f'Moyenne={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle=':',  lw=1.5, label=f'Médiane={df[col].median():.1f}')
    ax.set_title(f'Distribution — {col}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Distributions des variables numériques', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS / 'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Sauvegardé :', RESULTS / 'eda_distributions.png')

## 5️⃣ Profil démographique — Genre

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Pie chart ────────────────────────────────────────────────────────────────
counts = df['Gender'].value_counts()
axes[0].pie(
    counts, labels=counts.index, autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c'], startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Répartition Homme / Femme', fontsize=13, fontweight='bold')

# ── Box plots Age & Income par Genre ─────────────────────────────────────────
df_melt = df[['Gender', 'Age', 'Income', 'Score']].melt(id_vars='Gender')
sns.boxplot(
    data=df[['Gender', 'Age', 'Income', 'Score']].assign(
        Income_k=df['Income'] / 1000
    ),
    x='Gender', y='Age', hue='Gender',
    palette=['#3498db', '#e74c3c'], ax=axes[1], legend=False
)
axes[1].set_title('Distribution Age par Genre', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'eda_gender.png', dpi=150, bbox_inches='tight')
plt.show()

## 6️⃣ Profil par tranche d'âge (équivalent saisonnalité mensuelle)

In [ ]:
# Découpage en tranches d'âge
bins   = [0, 25, 35, 45, 55, 65, 100]
labels = ['<25', '25-35', '35-45', '45-55', '55-65', '65+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

age_profile = df.groupby('AgeGroup', observed=True).agg(
    Nombre        = ('ID',     'count'),
    Revenu_moyen  = ('Income', 'mean'),
    Score_moyen   = ('Score',  'mean'),
).round(1)

display(age_profile)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenu moyen par tranche d'âge
axes[0].bar(age_profile.index, age_profile['Revenu_moyen'],
            color='steelblue', alpha=0.75, edgecolor='white')
axes[0].plot(age_profile.index, age_profile['Revenu_moyen'],
             color='darkred', marker='o', linewidth=2, zorder=3)
axes[0].set_title('Revenu moyen par tranche d\'âge', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tranche d\'âge')
axes[0].set_ylabel('Revenu moyen')
axes[0].grid(True, axis='y', alpha=0.3)

# Score moyen par tranche d'âge
axes[1].bar(age_profile.index, age_profile['Score_moyen'],
            color='#2ecc71', alpha=0.75, edgecolor='white')
axes[1].plot(age_profile.index, age_profile['Score_moyen'],
             color='darkred', marker='s', linewidth=2, zorder=3)
axes[1].set_title('Score moyen par tranche d\'âge', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tranche d\'âge')
axes[1].set_ylabel('Score moyen')
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle(f'Profil par tranche d\'âge — {CSV_PATH.stem}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'eda_age_profile.png', dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Matrice de corrélation & Pairplot

In [ ]:
# ── Heatmap corrélation ───────────────────────────────────────────────────────
df_num = df[['Age', 'Income', 'Score']].copy()
corr   = df_num.corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.3f', cmap='coolwarm',
    mask=mask, ax=ax, linewidths=0.5,
    vmin=-1, vmax=1, center=0,
    cbar_kws={'label': 'Corrélation de Pearson'}
)
ax.set_title('Matrice de Corrélation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Pairplot colorié par Genre ────────────────────────────────────────────────
pair = sns.pairplot(
    df[['Age', 'Income', 'Score', 'Gender']],
    hue='Gender',
    palette={'Male': '#3498db', 'Female': '#e74c3c'},
    diag_kind='kde', plot_kws={'alpha': 0.5}
)
pair.figure.suptitle('Pairplot — Age / Income / Score par Genre',
                     y=1.02, fontsize=13, fontweight='bold')
pair.figure.savefig(RESULTS / 'eda_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Pairplot sauvegardé')

## 8️⃣ Visualisation interactive (Plotly)

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── Scatter Income vs Score colorié par Genre ─────────────────────────────────
fig = px.scatter(
    df, x='Income', y='Score', color='Gender',
    symbol='Gender', opacity=0.7,
    color_discrete_map={'Male': '#3498db', 'Female': '#e74c3c'},
    hover_data=['Age'],
    title='Revenu vs Score — colorié par Genre',
    template='plotly_white'
)
fig.update_traces(marker=dict(size=6))
fig.show()

# ── Violin plots ─────────────────────────────────────────────────────────────
fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=['Âge', 'Revenu', 'Score'])
for i, col in enumerate(['Age', 'Income', 'Score'], start=1):
    for gender, color in [('Male', '#3498db'), ('Female', '#e74c3c')]:
        sub = df[df['Gender'] == gender][col]
        fig2.add_trace(
            go.Violin(y=sub, name=gender, line_color=color,
                      box_visible=True, meanline_visible=True,
                      showlegend=(i == 1)),
            row=1, col=i
        )
fig2.update_layout(
    title='Violin plots — Distribution par Genre et Variable',
    template='plotly_white', height=500, violinmode='group'
)
fig2.show()
fig2.write_html(str(RESULTS / 'eda_violin_interactive.html'))
print('✔ Violin plots sauvegardés')

## 9️⃣ Nettoyage reproductible → fichier prêt pour le clustering

**Règles appliquées** :
1. Suppression des doublons complets
2. Suppression des lignes avec NaN
3. Encodage `Gender` → `Gender_enc` (Male=1, Female=0)
4. Création colonne `AgeGroup` (tranches)
5. Fichier écrit dans `data/processed/` pour les notebooks suivants

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_clean = df.copy()

# 1) Doublons
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f'Doublons supprimés : {before - len(df_clean)}')

# 2) NaN
before = len(df_clean)
df_clean.dropna(inplace=True)
print(f'Lignes NaN supprimées : {before - len(df_clean)}')

# 3) Encodage Gender
le = LabelEncoder()
df_clean['Gender_enc'] = le.fit_transform(df_clean['Gender'])
print(f'Encodage : {dict(zip(le.classes_, le.transform(le.classes_)))}')

# 4) AgeGroup
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['<25', '25-35', '35-45', '45-55', '55-65', '65+'],
    right=False
)

# 5) Sauvegarde
out_path = DATA_PROCESSED / f'{CSV_PATH.stem}_clean.csv'
df_clean.to_csv(out_path, index=False)

print(f'\nFichier sauvegardé : {out_path.resolve()}')
print(f'Lignes : brut={len(df)} | nettoyé={len(df_clean)}')
print(f'Doublons restants : {df_clean.duplicated().sum()} | NaN : {df_clean.isna().sum().sum()}')
display(df_clean.head())

## ✅ Synthèse EDA

| Observation | Détail |
|-------------|--------|
| **Dimensions** | À compléter après exécution |
| **Valeurs manquantes** | Aucune (à vérifier) |
| **Doublons** | À vérifier |
| **Outliers** | Quelques valeurs extrêmes sur Income (IQR×3) |
| **Corrélations** | Age–Score et Age–Income à analyser |
| **Genre** | Répartition ~50/50 (à vérifier) |

**Prochaine étape** → `02_preprocessing_clustering.ipynb`  
Normalisation + choix du k optimal (Elbow + Silhouette) + K-Means